# Proyecto Final Parte 1 — Análisis del Riesgo  
## BanBajío (BBAJIO) — Equipo 6

Este notebook documenta los cálculos utilizados para complementar el archivo HTML final del proyecto.

**Objetivos del notebook**
- Dejar trazabilidad de los márgenes construidos.
- Mostrar cómo se calculan los indicadores financieros solicitados.
- Reconstruir el EPS trimestral de 2019 a 2025.
- Generar una tabla y una gráfica de evolución del EPS.
- Distinguir entre cifras **reportadas** por BanBajío/BMV y métricas **construidas por el equipo**.

> Nota: las cifras monetarias se expresan en **millones de pesos mexicanos**, salvo que se indique lo contrario.


## Fuentes principales

1. Bolsa Mexicana de Valores — perfil e información financiera de BBAJIO:  
   https://www.bmv.com.mx/es/emisoras/perfil/BBAJIO-31431  
   https://www.bmv.com.mx/es/emisoras/informacionfinanciera/BBAJIO-31431-CGEN_CAPIT

2. BanBajío — Reporte Anual 2019:  
   https://www.bb.com.mx/cs/BanBajio/home/Inversionistas/Informaci%C3%B3n%20Financiera/Informacion-Anual/reporte-anual-2019.pdf

3. BanBajío — Reporte Anual 2022:  
   https://www.bb.com.mx/cs/BanBajio/home/Inversionistas/Informaci%C3%B3n%20Financiera/Informacion-Anual/Reporte-Anual-2022.pdf

4. BanBajío — Reporte Anual 2023:  
   https://www2.bb.com.mx/cs/BanBajio/home/Inversionistas/Informaci%C3%B3n%20Financiera/Informacion-Anual/Reporte-Anual-2023.pdf

5. BanBajío / BMV — Información financiera 4T 2025:  
   https://www.bmv.com.mx/docs-pub/infinbnc/infinbnc_1527714_2025-04_1.pdf

6. BanBajío / BMV — Información financiera 2T 2025:  
   https://www.bmv.com.mx/docs-pub/infinbnc/infinbnc_1481073_2025-02_1.pdf

Las cifras de EPS se reconstruyen a partir de utilidad neta trimestral y acciones en circulación.  


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.options.display.float_format = "{:,.4f}".format


# 1. Estados financieros 2023–2025

Para un banco, los conceptos de “margen bruto” y “EBITDA” no son tan naturales como en una empresa industrial.  
Por eso se trabaja con:
- partidas reportadas oficialmente;
- y, cuando la actividad lo pide, métricas construidas que se etiquetan como tales.


In [ ]:
# Partidas reportadas (millones de MXN)
# 2024 se mantiene como cifra comparable/reclasificada usada en el HTML final.

estados = pd.DataFrame({
    "Año": [2023, 2024, 2025],
    "Ingresos_por_intereses": [39938, 42901, 39496],
    "Margen_financiero": [22179, 22812, 21529],
    "Resultado_operacion": [15061, 14574, 12406],
    "Resultado_neto": [11013, 10689, 9079],
    # Ingreso operativo bancario construido a partir de:
    # margen financiero + comisiones netas + intermediación + otros ingresos/egresos operativos
    "Ingreso_operativo_bancario": [24894, 26000, 24965]
})

estados


## 1.1 Márgenes construidos

### Margen bruto bancario
\[
\text{Margen bruto bancario}
=
\frac{\text{Margen financiero}}
{\text{Ingresos por intereses}}
\times 100
\]

Esta razón intenta medir cuánto queda de los ingresos por intereses después del costo financiero del fondeo.

### Margen operativo
\[
\text{Margen operativo}
=
\frac{\text{Resultado de operación}}
{\text{Ingreso operativo bancario}}
\times 100
\]

### Margen neto
\[
\text{Margen neto}
=
\frac{\text{Resultado neto}}
{\text{Ingreso operativo bancario}}
\times 100
\]

Estas tres razones son **construidas por el equipo** para la actividad; no se presentan como indicadores oficiales de BanBajío.


In [ ]:
estados["Margen_bruto_bancario_%"] = (
    estados["Margen_financiero"] / estados["Ingresos_por_intereses"] * 100
)

estados["Margen_operativo_%"] = (
    estados["Resultado_operacion"] / estados["Ingreso_operativo_bancario"] * 100
)

estados["Margen_neto_%"] = (
    estados["Resultado_neto"] / estados["Ingreso_operativo_bancario"] * 100
)

margenes = estados[[
    "Año",
    "Margen_bruto_bancario_%",
    "Margen_operativo_%",
    "Margen_neto_%"
]].copy()

margenes


### Ejemplo 2025

\[
\text{Margen bruto bancario}
=
\frac{21{,}529}{39{,}496}
\times 100
\approx 54.51\%
\]

\[
\text{Margen operativo}
=
\frac{12{,}406}{24{,}965}
\times 100
\approx 49.69\%
\]

\[
\text{Margen neto}
=
\frac{9{,}079}{24{,}965}
\times 100
\approx 36.37\%
\]

La caída del margen operativo y neto entre 2023 y 2025 muestra que una menor proporción del ingreso operativo termina convirtiéndose en utilidad.


# 2. Indicadores financieros 2025

Se distingue entre indicadores reportados y métricas construidas.

- **ROE 2025:** 19.4% — reportado.
- **ROA 2025:** 2.4% — reportado.
- **ICAP 2025:** 15.46% — reportado.
- **Razón de apalancamiento regulatoria:** 10.66% — reportada.
- **Deuda/Capital:** construida usando deuda financiera estricta, no depósitos.
- **Pasivo/Capital:** construida para mostrar el apalancamiento contable total.
- **Deuda/EBITDA y Margen EBITDA:** proxies académicos; no son métricas bancarias estándar.


In [ ]:
# Datos 2025 (millones de MXN)
capital_contable = 48193
pasivo_total = 344554
prestamos_interbancarios = 52917
resultado_operacion = 12406
depreciacion_amortizacion = 947
ingreso_operativo_bancario = 24965

roe_2025 = 19.4
roa_2025 = 2.4
icap_2025 = 15.46
apalancamiento_reg_2025 = 10.66

deuda_capital = prestamos_interbancarios / capital_contable
pasivo_capital = pasivo_total / capital_contable

ebitda_proxy = resultado_operacion + depreciacion_amortizacion
deuda_ebitda_proxy = prestamos_interbancarios / ebitda_proxy
margen_ebitda_proxy = ebitda_proxy / ingreso_operativo_bancario * 100

indicadores = pd.DataFrame({
    "Indicador": [
        "ROE",
        "ROA",
        "ICAP",
        "Razón de apalancamiento regulatoria",
        "Deuda financiera / Capital",
        "Pasivo total / Capital",
        "Deuda financiera / EBITDA proxy",
        "Margen EBITDA proxy"
    ],
    "Valor": [
        roe_2025,
        roa_2025,
        icap_2025,
        apalancamiento_reg_2025,
        deuda_capital,
        pasivo_capital,
        deuda_ebitda_proxy,
        margen_ebitda_proxy
    ],
    "Tipo": [
        "Reportado",
        "Reportado",
        "Reportado",
        "Reportado",
        "Construido",
        "Construido",
        "Proxy académico",
        "Proxy académico"
    ]
})

indicadores


## Interpretación de Deuda/Capital

La observación de la profesora es importante: en un banco, los depósitos de clientes son parte normal del negocio de intermediación y no deben tratarse automáticamente como “deuda corporativa”.

Por eso se presentan dos razones:

\[
\text{Deuda financiera/Capital}
=
\frac{52{,}917}{48{,}193}
\approx 1.10x
\]

\[
\text{Pasivo total/Capital}
=
\frac{344{,}554}{48{,}193}
\approx 7.15x
\]

La primera usa préstamos interbancarios y otros organismos como deuda financiera estricta.  
La segunda incluye todos los pasivos y por eso debe interpretarse como **apalancamiento contable**, no como deuda financiera convencional.


# 3. EPS trimestral 2019–2025

Fórmula:

\[
EPS_t =
\frac{\text{Utilidad neta del trimestre } t}
{\text{Promedio ponderado de acciones en circulación}}
\]

Para mantener la serie homogénea se utiliza un denominador de **1,189.931687 millones de acciones Serie O**.

> La serie se construye con utilidad neta trimestral reportada. La cifra anual oficial de EPS puede diferir ligeramente por el promedio ponderado exacto, reclasificaciones y ajustes de cierre.


In [ ]:
acciones_millones = 1189.931687

utilidad_trimestral = {
    2019: [1410, 1412, 1389, 1402],
    2020: [1238, 705, 759, 735],
    2021: [945, 1070, 1250, 1549],
    2022: [1526, 1811, 2109, 2626],
    2023: [2691, 2782, 2755, 2785],
    2024: [2779, 2847, 2525, 2538],
    2025: [2491, 2166, 2267, 2155],
}

filas = []
for anio, utilidades in utilidad_trimestral.items():
    for trimestre, utilidad in enumerate(utilidades, start=1):
        eps = utilidad / acciones_millones
        filas.append({
            "Año": anio,
            "Trimestre": f"Q{trimestre}",
            "Utilidad_neta_millones_MXN": utilidad,
            "Acciones_millones": acciones_millones,
            "EPS_MXN_por_accion": eps
        })

eps_df = pd.DataFrame(filas)
eps_df


### Ejemplo de cálculo

Para Q1 2025:

\[
EPS_{Q1,2025}
=
\frac{2{,}491}{1{,}189.931687}
=
2.0934
\]

Para Q2 2025:

\[
EPS_{Q2,2025}
=
\frac{2{,}166}{1{,}189.931687}
=
1.8203
\]


In [ ]:
# Resumen anual: suma del EPS trimestral reconstruido
eps_anual = (
    eps_df.groupby("Año", as_index=False)["EPS_MXN_por_accion"]
    .sum()
    .rename(columns={"EPS_MXN_por_accion": "EPS_anual_reconstruido"})
)

eps_anual


In [ ]:
# Gráfica del EPS trimestral
eps_df["Periodo"] = eps_df["Año"].astype(str) + "-" + eps_df["Trimestre"]

plt.figure(figsize=(12, 5))
plt.plot(eps_df["Periodo"], eps_df["EPS_MXN_por_accion"], marker="o")
plt.title("BanBajío: EPS trimestral reconstruido, 2019–2025")
plt.xlabel("Periodo")
plt.ylabel("EPS (MXN por acción)")
plt.xticks(rotation=60)
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()


## Interpretación del EPS

La serie completa permite observar tres etapas:

1. **2019–2020:** deterioro durante la pandemia, con un mínimo cercano a $0.59 por acción en Q2 2020.
2. **2021–2023:** recuperación sostenida; el EPS supera $2 por acción en varios trimestres.
3. **2024–2025:** moderación de la rentabilidad por acción.

En 2025 el EPS reconstruido pasa de aproximadamente **$2.09 en Q1** a **$1.81 en Q4**, consistente con un entorno de menor margen financiero.

La conclusión correcta no es que el EPS sea “estable” en sentido estricto, sino que se mantuvo positivo durante todos los trimestres y mostró sensibilidad al ciclo económico, las provisiones y las tasas de interés.


# 4. Resumen de trazabilidad

| Elemento | Tipo | Procedimiento |
|---|---|---|
| Ingresos por intereses | Reportado | Estado de resultados |
| Margen financiero | Reportado | Estado de resultados |
| Resultado de operación | Reportado | Estado de resultados |
| Resultado neto | Reportado | Estado de resultados |
| Margen bruto bancario | Construido | Margen financiero / ingresos por intereses |
| Margen operativo | Construido | Resultado de operación / ingreso operativo bancario |
| Margen neto | Construido | Resultado neto / ingreso operativo bancario |
| ROE / ROA | Reportado | Reportes de BanBajío |
| Deuda/Capital | Construido | Préstamos interbancarios / capital contable |
| Pasivo/Capital | Construido | Pasivo total / capital contable |
| Deuda/EBITDA | Proxy | Deuda financiera / EBITDA proxy |
| Margen EBITDA | Proxy | EBITDA proxy / ingreso operativo bancario |
| EPS trimestral | Construido | Utilidad neta trimestral / acciones |

Este esquema permite distinguir claramente qué indicadores vienen directamente de los estados financieros y cuáles fueron construidos para el análisis académico.


# 5. Conclusión del notebook

Los cálculos apoyan la caracterización de BanBajío como un banco rentable y capitalizado, pero con una moderación en rentabilidad durante 2024–2025. La revisión de los márgenes muestra presión principalmente después del margen financiero, mientras que el EPS trimestral confirma una desaceleración frente a los máximos de 2023.

La exposición al crédito empresarial, la sensibilidad a tasas y la estructura de fondeo son factores que deben conectarse con el análisis posterior del riesgo de mercado de la acción BBAJIO.
